In [1]:
!pip install tf_keras

Defaulting to user installation because normal site-packages is not writeable


In [2]:
import tf_keras as keras
import pandas as pd
import h5py

with h5py.File('../resources/movetimes-with-pieces.h5', 'r') as f:
    pieces = f['pieces'][:]   # (N, 12, 8, 8)
    scalars = f['scalars'][:] 

df = pd.DataFrame(scalars, columns=['initial_time', 'increment', 'time_remaining_for_this_position', 'time_opponent_took', 'number_of_legal_moves', 'in_check', 'time_taken_for_this_position'])
df['pieces'] = list(pieces)  # each cell is a (12, 8, 8) ndarray

In [3]:
from sklearn.model_selection import train_test_split

nontest, test = train_test_split(df, test_size = 0.1, random_state = 42)

train, val = train_test_split(nontest, test_size = 0.1, random_state = 42)
# given the size of the datatest, cross-validation is unnecessary to maintain a large training dataset.

# 'time_taken_for_this_position' is the target feature, so we'll take it out and put it in a variables called y
def X_y_split(dataframe):
    X = dataframe.drop('time_taken_for_this_position', axis=1)
    y = dataframe['time_taken_for_this_position']
    return (X, y)

train_X, train_y = X_y_split(train)
val_X, val_y = X_y_split(val)

In [4]:
train_X.info()

<class 'pandas.core.frame.DataFrame'>
Index: 2820328 entries, 744012 to 3336675
Data columns (total 7 columns):
 #   Column                            Dtype  
---  ------                            -----  
 0   initial_time                      float32
 1   increment                         float32
 2   time_remaining_for_this_position  float32
 3   time_opponent_took                float32
 4   number_of_legal_moves             float32
 5   in_check                          float32
 6   pieces                            object 
dtypes: float32(6), object(1)
memory usage: 107.6+ MB


In [5]:
def split_training_data_into_pieces_and_others(data):
    pieces_inputs = np.stack(data['pieces'].values).astype(np.float32)
    other_inputs = data.drop('pieces', axis=1).values.astype(np.float32)
    return (pieces_inputs, other_inputs)

In [6]:
from tf_keras import layers
import numpy as np

# Input for the pieces feature. This feature will go through convolutions before rejoining the other features
pieces_inputs = keras.Input(shape=(12, 8, 8), name="pieces_inputs")

# Convolutional layers
x = layers.Conv2D(64, kernel_size=3, activation="relu", data_format="channels_first")(pieces_inputs)
x = layers.Conv2D(128, kernel_size=3, activation="relu", data_format="channels_first")(x)
x = layers.Conv2D(512, kernel_size=4, activation="relu", data_format="channels_first")(x)
x = layers.Flatten()(x)

# Other features (initial_time, increment, time_remaining_for_this_position, etc.
other_inputs = keras.Input(shape=(6,), name="other_inputs")

# Merge and dense layers
combined = layers.Concatenate()([x, other_inputs])
combined = layers.Dense(32, activation="relu")(combined)
combined = layers.Dense(32, activation="relu")(combined)
output = layers.Dense(1, name="output")(combined)

model = keras.Model(inputs=[pieces_inputs, other_inputs], outputs=output)
model.summary()


Model: "model"
__________________________________________________________________________________________________
 Layer (type)                Output Shape                 Param #   Connected to                  
 pieces_inputs (InputLayer)  [(None, 12, 8, 8)]           0         []                            
                                                                                                  
 conv2d (Conv2D)             (None, 64, 6, 6)             6976      ['pieces_inputs[0][0]']       
                                                                                                  
 conv2d_1 (Conv2D)           (None, 128, 4, 4)            73856     ['conv2d[0][0]']              
                                                                                                  
 conv2d_2 (Conv2D)           (None, 512, 1, 1)            1049088   ['conv2d_1[0][0]']            
                                                                                             

In [7]:
model.compile(optimizer=keras.optimizers.AdamW(weight_decay=0.004), loss="mse")

train_X_pieces, train_X_others = split_training_data_into_pieces_and_others(train_X)
val_X_pieces, val_X_others = split_training_data_into_pieces_and_others(val_X)

model_checkpoint_callback = keras.callbacks.ModelCheckpoint(
    filepath='../resources/checkpoint.weights.h5',
    monitor='loss',
    mode='max',
    save_weights_only=True,
    save_best_only=True)

history = model.fit(
    {"pieces_inputs": train_X_pieces, "other_inputs": train_X_others},
    y=train_y,
    epochs=20,
    batch_size=32,
    validation_data=({"pieces_inputs": val_X_pieces, "other_inputs": val_X_others}, val_y),
    callbacks=[keras.callbacks.EarlyStopping(monitor='loss', patience=2), model_checkpoint_callback],
)

Epoch 1/20

88136/88136 [==============================] - 2562s 29ms/step - loss: 57.2968 - val_loss: 52.1848
Epoch 2/20
88136/88136 [==============================] - 2548s 29ms/step - loss: 55.6371 - val_loss: 51.8881
Epoch 3/20
88136/88136 [==============================] - 2533s 29ms/step - loss: 55.2360 - val_loss: 52.1803
Epoch 4/20
88136/88136 [==============================] - 2539s 29ms/step - loss: 55.1724 - val_loss: 52.3594
Epoch 5/20
88136/88136 [==============================] - 2544s 29ms/step - loss: 55.0395 - val_loss: 52.0502
Epoch 6/20
88136/88136 [==============================] - 2524s 29ms/step - loss: 54.9800 - val_loss: 53.7496
Epoch 7/20
88136/88136 [==============================] - 2638s 30ms/step - loss: 55.0493 - val_loss: 51.8714
Epoch 8/20
88136/88136 [==============================] - 2609s 30ms/step - loss: 54.9684 - val_loss: 51.7652
Epoch 9/20
88136/88136 [==============================] - 2608s 30ms/step - loss: 54.9583 - val_loss: 51.9308
Epoch 10/

In [8]:
# Result: not significantly better than the previous model

In [9]:
import h5py 

model.save('weights-attempt3', save_format='h5')

model_json = model.to_json()
with open("model-attempt3.json", "w") as json_file:
    json_file.write(model_json)

C:\Users\Ainan Cawley\AppData\Roaming\Python\Python313\site-packages\tf_keras\src\engine\training.py:3098: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native TF-Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(
